In [2]:
###== Python Imports ##############
import tidy3d as td
import numpy as np
import pandas as pd
import math

import matplotlib.pyplot as plt
import plotly.graph_objects as go
# %matplotlib widget


###== Tidy3d Imports ###############
import tidy3d.web as web
print(td.__version__)

import os
# api_key = os.environ.get("TIDY3D_API_KEY")

from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv("TIDY3D_API_KEY")

from getpass import getpass
api_key = getpass("Enter your API key: ")
# web.configure("_blank_")
web.configure(api_key)

2.11.1


Enter your API key:  ········


Configuration saved successfully.


In [41]:
web.test()

11:37:55 Malay Peninsula Standard Time Authentication configured successfully!

In [3]:

save_dir = '/Users/Howfishy/Documents/0. tidy3d/May10 all gaps/'
# save_dir = '/app/local_project/18Feb Gap/'  
os.makedirs(save_dir, exist_ok=True)

# Definitions

In [5]:
#######################  Global Constants ####################################
##==  Define frequency range, run_time, auto-termination, apodiz, GridSpec, BoundarySpec ============

lambda_min, lambda_max = 0.056, 0.690                        # 22eV to 2eV
freq_max, freq_min     = td.C_0/lambda_min, td.C_0/lambda_max

freqs      = np.linspace(freq_min, freq_max, 201)  # 201 frequency points
freq_0     = (freq_max + freq_min)  / 2            # source centre 
freq_width = (freq_max - freq_min)                 # source witdth 


###########################################################
########### Material definitions ##########################
###########################################################
Air = td.Medium(
    name = 'Air',
    permittivity = 1.000,
)

Si_Palik_4_poles = td.PoleResidue(
    name = 'Si_Palik_4-poles',
    frequency_range = [freq_min, freq_max],
    eps_inf = 1.0201315339282613,
    poles = [[(-230298116405116  -8124044914089037j), (-54468577807527.086 + 282082050889749.9j)], 
             [(-1689702290399004 -6497791373591796j), (22171856177039732   + 49896720261586410j)], 
             [(-222926848767196.12-6425424240903650j), (1710560534805847.5 + 3334591681401622.5j)], 
             [(-742335363792465.5 -6679004751836012j), (-22116228281991896 - 5135107205058006j)]]
)


###########################################################
##############  Simulation settings  ######################
###########################################################


############# Grid discretization ################
GridSpec = td.GridSpec(
        grid_x = td.UniformGrid(dl = 0.002),  # 2nm on all sides
        grid_y = td.UniformGrid(dl = 0.002), 
        grid_z = td.UniformGrid(dl = 0.002), 
        wavelength = 0.2 )


############ Boundary (PML on all faces; tweaked kappa/alpha for stability) ################
_pml_params = td.PMLParams(
    kappa_min   = 1, kappa_max  = 3,
    alpha_order = 1, alpha_max  = 0,
)

BoundSpec = td.BoundarySpec(
    x = td.Boundary(
        # plus  =td.Periodic(), 
        # minus =td.Periodic()),  # infinite array
        plus  =td.PML(parameters=_pml_params), 
        minus =td.PML(parameters=_pml_params)),
    y = td.Boundary(
        plus  =td.PML(parameters=_pml_params), 
        minus =td.PML(parameters=_pml_params)),
    z = td.Boundary(
        plus  =td.PML(parameters=_pml_params), 
        minus =td.PML(parameters=_pml_params)),
)

val_Apodization = td.ApodizationSpec()  #### set as Default = no apodization
# val_Apodization = td.ApodizationSpec(start = 0.4e-15, width = 0.4e-15)  # start = 0.4fs, ramp 0.4fs

In [4]:
# Planck's constant in Js
h = 6.62607015e-34
# Elementary charge in C
e = 1.602176634e-19

# Energy in eV = (frequency_hz * h) / e
def hz_to_ev(frequency_hz):
    '''Args: (float or np.ndarray)'''
    return (frequency_hz * h) / e

def ev_to_hz(energy_ev):
    return (energy_ev * e) / h

In [13]:
def check_run_time(cfg, freq_min):
    suggested = cfg['n_periods'] / freq_min
    
    print(f"[INFO] Suggested minimum run_time : {suggested:.3e} s")
    print(f"[INFO] Configured  run_time       : {cfg['run_time']:.3e} s")
    
    if cfg['run_time'] < suggested:
        print("[WARN] run_time is shorter than suggested minimum — consider increasing it.")

###########################################################################################

def recommend_sim_size(cfg, n_holes, lambda_max):
    """
    Compute and return a simulation box size that satisfies the PML distance rule.

    Parameters
    ----------
    radius          : float  cavity radius [µm]
    gap             : float  edge-to-edge gap between the two cavities [µm]
    thickness       : float  slab thickness [µm]
    lambda_max      : float  longest wavelength in the simulation [µm]
    pml_pad_factor  : float  minimum PML padding as a fraction of lambda_max
                             (default 0.5; toward 1.0 if the simulation diverges)
                             - For strong evanescent fields, even larger padding may be needed.
                             - Waveguides or structures feeding into the boundary should use td.inf.

    Returns
    -------
    size : list[float]  [Lx, Ly, Lz]
    """
 
    pad  = cfg['pml_pad_factor'] * lambda_max
    pitch            = cfg['diam'] + cfg['gap']
    
    geometry_span_x  = n_holes*pitch        # dimer spans (~2× pitch) in XY
    geometry_span_z  = cfg['thickness']
    
    Lx = geometry_span_x + 2*pad 
    Ly = pitch + 2*pad
    Lz = geometry_span_z  + 2*pad

    print(f"\n[PML CHECK]  lambda_max = {lambda_max:.4f} µm  |  pad = {pad:.4f} µm")
    print(f"             Recommended sim size: [{Lx:.4f}, {Ly:.4f}, {Lz:.4f}] µm")
    print(f"             (pml_pad_factor = {cfg['pml_pad_factor']}  →  "
          f"PML at least {pad*1e3:.1f} nm from structures)\n")
    
    return [Lx, Ly, Lz]
    # return [pitch, Ly, Lz] # for infinite array

In [7]:
def make_structures(cfg, n_holes):
    """
    Build simulation structures: infinite slab + (n) cylindrical air cavities.
    # 'hole_at_origin' = The middle cavity is at (0,0,0) always. (both even or odd_n)
    # (or 'span_centered' = the midpoint of entie array) # for odd_n

    """
    radius    = cfg['diam'] * 0.5
    thickness = cfg['thickness']
    pitch     = cfg['gap'] + cfg['diam']
    # --- Centering mode ---
    centering = cfg.get('array_centering', 'hole_at_origin')  # (one hole always at x=0) is default if cfg not specified
    
    # Create a list of all cylinder geometries first          
    geoms = []
    
    if centering == 'hole_at_origin':
        
        start_idx = -(n_holes // 2)
        for i in range(n_holes):
            x_pos = (start_idx + i) * pitch
            geoms.append( td.Cylinder(center=(x_pos, 0, 0), 
                                      radius=radius, 
                                      length=thickness))
    
    elif centering == 'span_centered':
        
        start_x = -(n_holes - 1) * pitch  / 2.0
        for i in range(n_holes):
            x_pos = start_x + (i * pitch)
            geoms.append(td.Cylinder(center=(x_pos, 0, 0), 
                                     radius=radius, 
                                     length=thickness))
    else:
        raise ValueError(f"Unknown array_centering mode: '{centering}'. "
                         f"Choose 'hole_at_origin' or 'span_centered'.")
    
    # Better to Bundle them up =>  (ONE td.GeometryGroup structure) is more efficient than (list of N cylinders)
    Hole_array = td.Structure(
        name="Hole_Array",
        geometry=td.GeometryGroup(geometries=geoms),
        medium=Air
    )
    
    # The slab remains a separate structure
    Slab = td.Structure(
        name     = 'Slab',
        geometry = td.Box(center=(0, 0, 0), size=[td.inf, td.inf, thickness]),
        medium   = Si_Palik_4_poles,
    )

    return [Slab, Hole_array]

In [15]:
def _random_points_in_cylinder(cfg, center, n_points):
    """Uniform random sampling inside a cylinder"""
    rad       = cfg['diam']  * 0.5
    length    = cfg['thickness'] * 2
    cx, cy, cz = center  

    # Random radius (with sqrt for uniform area distribution)
    points = []
    for _ in range(n_points):
        spacing = 0.05 * rad  ### maximum radius used is radius - spacing
        r = (rad-spacing) * np.sqrt(np.random.rand())    # points not touching cylinder walls
        theta = 2*np.pi * np.random.rand()

        # Then Use random small fluctuations
        x = cx + r * np.cos(theta)
        y = cy + r * np.sin(theta)
        z = cz + (np.random.rand() - 0.5) * length # Uniformly along z
        points.append([x, y, z])
        
    return points

def _vertical_dipoles_in_cylinder(cfg, center, n_points):
    """Evenly-spaced vertical line of dipoles at the cylinder axis."""
    rad       = cfg['diam'] * 0.5
    length = cfg['thickness'] * 2      # dipole_thickness = twice * slab_thickness
    cx, cy, cz = center    # cz is the middle of slab
    spacing = 0.000    
    
    points = []
    # Calculate TOTAL vertical spacing between dipoles
    dipole_thickness = length - 2*spacing # Total available length 
    
    if n_points == 1:
        # Single dipole at center
        points.append([cx, cy, cz])
    else:
        # Multiple dipoles evenly spaced along z-axis
        z_spacing = dipole_thickness / (n_points - 1)
        for i in range(n_points):
            x = cx  # Center x position UNCHANGED
            y = cy  # Center y position UNCHANGED
            
            # Start from bottom (cz - dipole_thickness/2) and go up
            bottom_most = cz - dipole_thickness/2
            z = bottom_most   + (i * z_spacing)
            points.append([x, y, z])
    
    return points    

def make_dipole_points(cfg, cavity_center):
    """
    Generate dipole positions according to cfg['dipole_mode'].
    'both_vertical'     : random fill in both cavities (dimer excitation)   # nope
    'both_random'       : random fill in both cavities (dimer excitation)   # nope
    'vertical_n1' : vertical line in primary cavity only
    'random_n1'   : random fill in primary cavity only
    
    Returns All_points : entire list of [x, y, z]
    """
    radius           = 0.5 * cfg['diam']
    n1               = cfg['n1']
    n2               = cfg['n2']
    mode             = cfg['dipole_mode']

    if mode == 'vertical_n1':
        All_points = _vertical_dipoles_in_cylinder( cfg, cavity_center, n1)
    
    elif mode == 'random_n1':
        np.random.seed(69)   # remove arg for non-reproducible runs (69 = 'fixed' randomisation)
        All_points = _random_points_in_cylinder(cfg, cavity_center, n1)
    
    # elif mode == 'both_vertical':
    #     np.random.seed()
    #     v_pts1 = _vertical_dipoles_in_cylinder( cfg, cavity_center, n1)
    #     v_pts2 = _vertical_dipoles_in_cylinder( cfg, paired_cavity_center, n2)
    #     All_points = v_pts1 + v_pts2

    # elif mode == 'both_random':
    #     np.random.seed()
    #     r_pts1 = _random_points_in_cylinder(cfg, cavity_center, n1)
    #     r_pts2 = _random_points_in_cylinder(cfg, paired_cavity_center, n2)
    #     All_points = r_pts1 + r_pts2

    else:
        raise ValueError(f"Unknown dipole_mode '{mode}'. Choose 'vertical_n1' OR 'random_n1'.") # or  'both_vertical','both_random'.

    print(f"[INFO] dipole_mode='{mode}'  →  {len(All_points)} dipoles generated.")
    return All_points

In [9]:
def make_sources(cfg, All_points):
    """
    Build PointDipole sources from a list of positions.

    Phase behaviour is controlled by cfg['use_phase']:
      False → all dipoles share the same Gaussian pulse (zero phase)
      
      True  → each dipole gets a phase offset matching electron arrival time  ############# 
              ## (CTR / Smith-Purcell style excitation)
              (electron travels in -z direction).
              First 3 dipoles printed for debugging.
    """
    pol_value  = cfg['pol_value']
    
    use_phase  = cfg['use_phase']
    e_vel      = cfg['electron_vel_c'] * td.constants.C_0

    z_positions = [pos[2] for pos in All_points] # [X,Y,Z] = [0,1,2]
    z_max       = max(z_positions)               # electron arrives here first (top)
    
    dipole_sources = []
    print(f"[INFO] use_phase={use_phase}")
    for i, pos in enumerate(All_points):
        z_pos      = pos[2]
        d_traveled = z_max - z_pos              # electron travels in -z direction
        t_arrival  = d_traveled / e_vel
        p_offset   = 2*np.pi * freq_0*t_arrival if use_phase   else 0.0  # p_offset = 0.0 when (use_phase = False)

        source_time = td.GaussianPulse(
            freq0  = freq_0,
            fwidth = freq_width,
            phase  = p_offset,
        )

        dip = td.PointDipole(
            name         = f'pointdipole_{i}',
            center       = pos,
            polarization = pol_value,
            source_time  = source_time,
        )

        if i < 3 and use_phase:   # debug: print first 3 dipoles
            print(f"  Dipole {i:3d}: z={z_pos:+.4f} µm | "
                  f"t_arrival={t_arrival:.3e} s | phase={p_offset:.4f} rad")

        dipole_sources.append(dip)

    print(f"[INFO] Dipole sources created →  {len(dipole_sources)}.")
    return dipole_sources

In [51]:
## ** new plane_wave function **
def make_plane_wave_source(cfg, size_sim):    #### included just in case
    """
    Single plane wave propagating in -z direction (top-down).
    pol_angle: 0 → x-polarised,  pi/2 → y-polarised.
    """
    ANGLE = cfg['pol_angle']
    z_top = cfg['thickness'] * 2   # just inside top of simulation Box

    source = td.PlaneWave(
        name        = 'plane_wave',
        center      = [0, 0, z_top],
        size        = [td.inf, td.inf, 0],  # infinite xy
        direction   = '-',               # propagating in -z
        pol_angle   = ANGLE,  # 0 → along x,  pi/2 → along y
        source_time = td.GaussianPulse(freq0   = freq_0,
                                       fwidth  = freq_width,
        ),
    )
    print(f"[INFO] source_type='plane_wave'  |  pol_angle={cfg['pol_angle']} rad  |  z={z_top:.4f} µm")
    return [source]

In [10]:
def make_monitors(cfg, monitor_size_in, monitor_size_out, monitor_names):
    """
    Build field monitors from an explicit list of monitor names.

    Parameters
    ----------
    monitor_names : list of str
        Names to include, chosen from the monitor_table below.
    """
    lx, ly    = monitor_size_in
    lh, lv    = monitor_size_out
    
    rad       = cfg['diam'] * 0.5
    thickness = cfg['thickness']
    pitch     = cfg['gap'] + cfg['diam']

    # Unchanged values
    FIELDS = ['Ex', 'Ey', 'Ez', 'Hx', 'Hy', 'Hz']
    shared = dict(freqs=freqs, 
                  fields=FIELDS, 
                  apodization=val_Apodization)

    # ── Monitor table: (name, center, size) ───────────────────────────────────
    monitor_table = [
        # name                      center                                   size
        ('in_plane_slice0',        [0, 0,  0],                             [lx , ly  , 0]   ),
        ('in_plane_slice4.5',      [0, 0, -(4.5*0.10)*thickness],          [lx , ly  , 0]   ),
        ('out_plane',              [0, 0,  0],                             [lh ,  0,   lv]),
        
        ('xy_bottom_slice0.5',     [0, 0, -thickness/2 - (0.5*0.10)*thickness],  [lx , ly, 0] ),
        ('xy_bottom_slice2',       [0, 0, -thickness/2 - ( 2 *0.10)*thickness],  [lx , ly, 0] ),
        ('xy_bottom_slice4',       [0, 0, -thickness/2 - ( 4 *0.10)*thickness],  [lx , ly, 0] ),
        ('xy_bottom_slice8',       [0, 0, -thickness/2 - ( 8 *0.10)*thickness],  [lx , ly, 0] ),
        ('xy_bottom_slice14',      [0, 0, -thickness/2 - ( 14*0.10)*thickness],  [lx , ly, 0] ),
        

        # ('in_plane_Left',          [-0.25*lx, 0, 0],                       [0.5*lx ,    l   ,   0]  ),
        # ('in_plane_Right',         [+0.25*lx, 0, 0],                       [0.5*lx ,    ly  ,   0]  ),
        # ('ZX_plane_n1',            [-0.25*lh, 0, 0],                       [0.5*lh ,    0   ,   lv] ),
        # ('ZX_plane_n2',            [+0.25*lh, 0, 0],                       [0.5*lh ,    0   ,   lv] ),
        # ('ZY_plane_n1',            [+0.5*pitch, 0, 0],                     [0      , 0.75*lh,   lv] ),
        # ('ZY_plane_n2',            [-0.5*pitch, 0, 0],                     [0      , 0.75*lh,   lv] ),
    ]

    table_dict = {name: (center, size)   for name, center, size in monitor_table}

    unknown = [n for n in monitor_names   if n not in table_dict]
    if unknown:
        raise ValueError(f"Unknown monitor name(s): {unknown}. "
                         f"Available: {list(table_dict)}")

    monitors = [
        td.FieldMonitor(name   ='DFT'+ name, 
                        center =table_dict[name][0], 
                        size   =table_dict[name][1], 
                        **shared)
        for name in monitor_names
    ]

    print(f"[INFO] {len(monitors)} monitors created: {monitor_names}")
    return monitors

# Build simulations

In [54]:
# cfg for ANY 1D array => in X-axis direction
# fmt: off
cfg = dict(
    # ── Geometry ─────────────────────────────────────────────────────────────
    thickness       = 0.100,     # slab thickness  [µm]
    diam            = 0.070,     # cavity diameter [µm]
    gap             = 0.100,     # edge-to-edge gap between cavities [µm]
    
    # array_centering = 'hole_at_origin',  # 'hole_at_origin' → one hole always sits at x=0 (for both n_odd & n_even )
    array_centering = 'span_centered',  # 'span_centered'  → midpoint of entire array centred at x=0 
    
    num_cavities    = 2,         # number of holes in the 1D array
    # num_cavities = 15,

    source_type     = 'dipole',      # 'dipole'     → EELS-style point dipoles in cavity
    # source_type     = 'plane_wave',  # (included but not used)
                                      ######'plane_wave' → infinite plane wave from above, propagating -z direction 

    # pol_angle = 0,     ###==  Include this if using plane_wave  ===###
                         # plane wave polarisation angle [rad]  (dipole-only runs: ignored)
                         # 0 → x-polarised,  pi/2 → y-polarised
    
    # ── Dipole sources ───────────────────────────────────────────────────────
    n1              = 160,       # dipoles in primary cavity
    n2              = 160,       # dipoles in paired cavity                  # not used
    pol_value       = 'Ez',      # polarisation:  'Ex' | 'Ey' | 'Ez'
    
    dipole_mode     = 'vertical_n1',        # vertical fill in primary cavity only
    # dipole_mode     = 'random_n1',        # random fill in primary cavity only

    
    ####### dipole_mode     = 'both_vertical',    # vertical fill in both cavities (dimer excitation)  # nope
    ###### dipole_mode      = 'both_random',      # random fill in both cavities (dimer excitation)    # nope

    dipole_position = 'center',  ###### preset of BEAM_probe within CAVITY:
                     # 'center'     → on-axis centre of hole
                     # 'edge'       → +x edge  (+0.80r)
                     # 'close_edge' → -x edge  (-0.80r)
                     # 'y_edge'     → -y edge  (-0.80r in y)
                     # 'solid_edge' → gap midpoint into SLAB (-0.5 pitch)
    
    # ── dipole Phase ────────────────────────────────────────────────────────────────
    use_phase       = True,     # True  → phase-shifted excitation
                                 # False → all dipoles fire in phase
    electron_vel_c  = 0.50,      # electron speed as fraction of c  (80 keV ≈ 0.50)
                                 # 60 keV→0.45  100 keV→0.55  200 keV→0.70  300 keV→0.78

    # ── Monitors ─────────────────────────────────────────────────────────────
    monitors        = [ 'in_plane_slice0',
                        'in_plane_slice4.5',
                        'out_plane',
                         # 'xy_bottom_slice0.5'  'xy_bottom_slice2'  'xy_bottom_slice4'
                         # 'xy_bottom_slice8'    'xy_bottom_slice14'

                         # 'in_plane_Left'       'in_plane_Right'
                         # 'ZX_plane_n1'         'ZX_plane_n2'
                         # 'ZY_plane_n1'         'ZY_plane_n2'
                      ],

    # ── PML padding (auto-sizes simulation box) ───────────────────────────────
    pml_pad_factor  = 0.5,       # recommended ≥ 0.5 × lambda_max from any structure
                                 # increase if simulation diverges

    # ── Run control ──────────────────────────────────────────────────────────
    n_periods       = 20,        # used to suggest minimum run_time
    run_time        = 6e-14,     # [s]  override if needed
    val_shutoff     = 1e-5,      # sim auto terminates if 100,000 times smaller than intial signal
                                         # 0 = disable (use for high-Q resonators), 
    val_norm        = 0,         #  source index, base_line pwoer normalisation for flux monitors (not important for field monitors)
)
# fmt: on

# ── PML-aware simulation box ──────────────────────────────────────────────────
size_sim = recommend_sim_size(cfg, cfg['num_cavities'], lambda_max)

# size_sim = [0.500, 0.500, 0.500]    ######  manual override if needed ############


# ── Monitor sizes (can manually override if needed) ───────────────────────────
## Monitor sizes smaller than simulation domain so (sides not touching for good simulation results)
monitor_size_in  = (0.75*size_sim[0]  , 0.75*size_sim[1])   # (Lx, Ly) for InPlane
monitor_size_out = (0.8*size_sim[0]  , 0.8*size_sim[2])   # (Lx, Lz) for OutPlane
# monitor_size_in  = (1.000, 0.600)
# monitor_size_out = (1.000, 0.800) 


[PML CHECK]  lambda_max = 0.6900 µm  |  pad = 0.3450 µm
             Recommended sim size: [1.0300, 0.8600, 0.7900] µm
             (pml_pad_factor = 0.5  →  PML at least 345.0 nm from structures)



In [53]:
# ── Automatically builds simulation from cfg ──────────────────────────────────────────────────────────

rad       = 0.5 * cfg['diam']
# thickness = cfg['thickness']
gap       = cfg['gap']
pitch     = 2 * rad + gap


# shift = 0               # hole_at_origin
# shift = 0.5 * pitch     # span_centered
shift = 0.5 * pitch if cfg['array_centering'] == 'span_centered' else 0

positions = {
    "center":     (0 + shift, 0, 0),
    "edge":       (0.80*rad + shift, 0, 0),
    "close_edge": (-0.80*rad + shift, 0, 0),
    "y_edge":     (0 + shift, -0.80*rad, 0),
    "solid_edge": (-0.5 * pitch + shift, 0, 0),
}

mode = cfg['dipole_position']
cavity_center = positions[mode]


# ── Assemble ──────────────────────────────────────────────────────────────────
# included plane_wave option in cfg
if cfg['source_type'] == 'plane_wave':
    points_All    = []          # not used, but kept for summary print
    Sources_All   = make_plane_wave_source(cfg, size_sim)
elif cfg['source_type'] == 'dipole':
    points_All    = make_dipole_points(cfg, cavity_center)
    Sources_All   = make_sources(cfg, points_All)
else:
    raise ValueError(f"Unknown source_type '{cfg['source_type']}'. "
                     f"Choose 'dipole' or 'plane_wave'.")

# ── Pass all information into td.Simulation ────────────────────────────────────

# points_all = make_dipole_points( cfg, cavity_center)
# Sources_All    = make_sources(       cfg, points_all)
Structures_All = make_structures(    cfg, cfg['num_cavities'] )  # 15 for 1D array
Monitors_All   = make_monitors(      cfg, monitor_size_in, monitor_size_out, cfg['monitors'])

sim = td.Simulation(
    size            = size_sim,
    run_time        = cfg['run_time'],
    shutoff         = cfg['val_shutoff'],
    normalize_index = cfg['val_norm'],
    medium          = Air,
    sources         = Sources_All,
    structures      = Structures_All,
    monitors        = Monitors_All,
    # version         = '2.9.1',
    boundary_spec   = BoundSpec,
    grid_spec       = GridSpec,
)

sim0 = sim.copy(update={'structures': []})


# ── Print summary ─────────────────────────────────────────────────────────────
print(
    f"\n{'─'*55}\n"
    f"  polarisation : {cfg['pol_value']}\n"
    f"  diameter     : {cfg['diam']*1e3:.0f} nm\n"
    f"  thickness    : {thickness*1e3:.0f} nm\n"
    f"  pitch        : {pitch*1e3:.0f} nm  (gap = {gap*1e3:.0f} nm)\n"
    f"  dipole_mode  : {cfg['dipole_mode']}  ({len(points_all)} dipoles)\n"
    f"  beam position: {cfg['dipole_position']}\n"
    f"  use_phase    : {cfg['use_phase']}  "
    f"(velocity = {cfg['electron_vel_c']}c)\n"
    f"  monitors used: {cfg['monitors']}\n"
    f"  sim size     : {[f'{s:.3f}' for s in size_sim]} µm\n"
    f"{'─'*55}"
)




# ── Visual inspection ─────────────────────────────────────────────────────────
sim.plot_3d()

# ── Auto-generate task name from config ───────────────────────────────────────
name = (
    f"{cfg['num_cavities']}holes_"

    f"Th{int(thickness * 1000)}"
    f"D{int(rad*2 * 1000)}"
    f"G{int(gap * 1000)}"
    
    f"_dipol{cfg['pol_value']}"                   # Can Remove to shorten file name
    f"_{cfg['dipole_mode']}"                      # Can Remove to shorten file name
    f"_{cfg['dipole_position']}"                  # Can Remove to shorten file name
    f"_{'withphase' if cfg['use_phase'] else 'nophase'}" # Can Remove to shorten file name
    # f"_{cfg['monitors']}"          # Removed to shorten file name
)

print(f"\n[INFO] Task name: {name}")

[INFO] source_type='plane_wave'  |  pol_angle=0 rad  |  z=0.2000 µm
[INFO] 3 monitors created: ['in_plane_slice0', 'in_plane_slice4.5', 'out_plane']

───────────────────────────────────────────────────────
  polarisation : Ez
  diameter     : 70 nm
  thickness    : 100 nm
  pitch        : 170 nm  (gap = 100 nm)
  dipole_mode  : vertical_n1  (160 dipoles)
  beam position: center
  use_phase    : True  (velocity = 0.5c)
  monitors used: ['in_plane_slice0', 'in_plane_slice4.5', 'out_plane']
  sim size     : ['1.030', '0.860', '0.790'] µm
───────────────────────────────────────────────────────



[INFO] Task name: 2holes_Th100D70G100_dipolEz_vertical_n1_center_withphase


# Estimate Cost

In [37]:
# Estimate cost of my simulation before running
# estimated_cost = td.web.estimate_cost(sim)
# print(f"Estimated simulation cost: {estimated_cost} credits")

from tidy3d.web.api.container import Job
job = web.Job(simulation=sim, 
              task_name="cost_estimate")
estimated_cost = job.estimate_cost()

13:10:57 Malay Peninsula Standard Time Created task 'cost_estimate' with        
                                       resource_id                              
                                       'fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c
                                       0' and task_type 'FDTD'.

                                       View task using web UI at                
                                       ]8;id=1881102;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\'https://tidy3d.simulation.cloud/workbenc]8;;\
                                       ]8;id=1881102;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\h?]8;;\]8;id=1881103;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\taskId]8;;\]8;id=1881102;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\=]8;;\]8;id=1881104;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\fdve]8;;\]8;id=1881102;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\-484deba8-b894-4d7a-8a4c-7f6]8;;\
                                       ]8;id=1881102;https://tidy3d.simulation.cloud/workbench?taskId=fdve-484deba8-b894-4d7a-8a4c-7f6dcd6023c0\dcd6023c0']8;;\.

                                       Task folder: ]8;id=1881107;https://tidy3d.simulation.cloud/folders/folder-b04f772b-b22f-4e19-a37d-8d56d622b095\'default']8;;\.

Output()

13:11:06 Malay Peninsula Standard Time Estimated FlexCredit cost: 0.798. Minimum
                                       cost depends on task execution details.  
                                       Use 'web.real_cost(task_id)' to get the  
                                       billed FlexCredit cost after a simulation
                                       run.

# Run simulation

In [49]:
######=========  RUN Simulations ===========####
###### ===============================#############


sim_data = web.run(
    sim, 
    task_name=name, 
    path=save_dir +name+'.hdf5',
    verbose=True)

sim_data0 = web.run(
    sim0, 
    task_name= name +'_empty', 
    path=save_dir +name+'_empty'+'.hdf5',
    verbose=True)

11:53:53 Malay Peninsula Standard Time Created task                             
                                       'center_2_dipolEz_Th100D70G100_vertical_n
                                       1_nophase_in_out' with resource_id       
                                       'fdve-a703056c-32dc-498b-96eb-618688e46f0
                                       a' and task_type 'FDTD'.

                                       View task using web UI at                
                                       ]8;id=5372012;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\'https://tidy3d.simulation.cloud/workbenc]8;;\
                                       ]8;id=5372012;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\h?]8;;\]8;id=5372013;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\taskId]8;;\]8;id=5372012;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\=]8;;\]8;id=5372014;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\fdve]8;;\]8;id=5372012;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\-a703056c-32dc-498b-96eb-618]8;;\
                                       ]8;id=5372012;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a703056c-32dc-498b-96eb-618688e46f0a\688e46f0a']8;;\.

                                       Task folder: ]8;id=5372016;https://tidy3d.simulation.cloud/folders/folder-b04f772b-b22f-4e19-a37d-8d56d622b095\'default']8;;\.

Output()

11:53:57 Malay Peninsula Standard Time Estimated FlexCredit cost: 0.214. Minimum
                                       cost depends on task execution details.  
                                       Use 'web.real_cost(task_id)' to get the  
                                       billed FlexCredit cost after a simulation
                                       run.

...

                                       status = postprocess

11:58:10 Malay Peninsula Standard Time status = success

11:58:12 Malay Peninsula Standard Time View simulation result at                
                                       ]8;id=5372035;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\'https://tidy3d.simulation.cloud/workbenc]8;;\
                                       ]8;id=5372035;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\h?]8;;\]8;id=5372036;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\taskId]8;;\]8;id=5372035;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\=]8;;\]8;id=5372037;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\fdve]8;;\]8;id=5372035;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\-8f4f1154-6dd4-4987-8a85-7ff]8;;\
                                       ]8;id=5372035;https://tidy3d.simulation.cloud/workbench?taskId=fdve-8f4f1154-6dd4-4987-8a85-7ffad8f159b0\ad8f159b0']8;;\.

Output()

11:59:50 Malay Peninsula Standard Time Loading results from                     
                                       \Users\Howfishy\Documents\0. tidy3d\May10
                                       all                                      
                                       gaps\center_2_dipolEz_Th100D70G100_vertic
                                       al_n1_nophase_in_out_empty.hdf5

### Workings are hidden